# Module 10 lab · Transformers and GPT

The code from every lesson of [this module](https://aicourses.dev/research/transformers), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 10.2 Self-attention, one head, step by step

Lets each position read any earlier one, which removes the MLP model’s fixed window.

[Read the lesson](https://aicourses.dev/research/transformers/self-attention)

### Scores, shares and a blend, by hand

1. **Score** each word by the dot product of the query with its key. Mary: 2 × 1 + 0 × 0 = 2. visited: 0. Paris: 2 × 0 + 0 × 1 = 0. she: 2 × 0.5 = 1.
2. **Share** the attention with softmax. $e^2 \approx 7.389$, $e^0 = 1$, $e^0 = 1$ and $e^1 \approx 2.718$ add up to 12.107, so the shares are 0.610, 0.083, 0.083 and 0.225.
3. **Blend** the words’ **values**, the messages they pass on, using those shares. If each value says which word it came from, the blend for “she” is 61% Mary, 22.5% itself and 8% each for the other two.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

words = ["Mary", "visited", "Paris", "she"]
K = torch.tensor([[1.0, 0.0],    # Mary: a person
                  [0.0, 0.0],    # visited: neither
                  [0.0, 1.0],    # Paris: a place
                  [0.5, 0.0]])   # she: a little person-like
q_she = torch.tensor([2.0, 0.0]) # she asks: "who is a person?"
scores = K @ q_she               # one score per word
weights = F.softmax(scores, dim=0)
for w, s, a in zip(words, scores.tolist(), weights.tolist()):
    print(f"{w:8} score {s:.1f}  weight {a:.3f}")
# Mary     score 2.0  weight 0.610
# visited  score 0.0  weight 0.083
# Paris    score 0.0  weight 0.083
# she      score 1.0  weight 0.225

### Why divide by the square root

Add up more products and a dot product gets bigger. For random lists of numbers with an average of 0 and a spread of 1, the spread of their dot product grows like the square root of their length.

In [ ]:
torch.manual_seed(0)
for d in [4, 64, 512]:
    q, k = torch.randn(10000, d), torch.randn(10000, d)
    dots = (q * k).sum(dim=1)
    print(d, round(dots.std().item(), 2),
          round((dots / d**0.5).std().item(), 2))
# 4 1.99 1.0
# 64 8.01 1.0
# 512 22.49 0.99

### One head in PyTorch

In [ ]:
torch.manual_seed(42)
T, C, hs = 4, 8, 4       # 4 tokens, 8 numbers each, head size 4
x = torch.randn(T, C)    # one embedding per token
Wq = torch.randn(C, hs) / C**0.5   # learned in a real model
Wk = torch.randn(C, hs) / C**0.5
Wv = torch.randn(C, hs) / C**0.5
q, k, v = x @ Wq, x @ Wk, x @ Wv   # each (4, 4)
scores = q @ k.T / hs**0.5         # (4, 4): every query, every key
weights = F.softmax(scores, dim=-1)  # each row adds up to 1
out = weights @ v                  # (4, 4): one blend per token
print(weights.round(decimals=2))
print(out.shape)                   # torch.Size([4, 4])

## 10.3 Causal masking and why GPT can’t peek

When guessing the next word, the model must not see the words that come after it. A mask hides them, and that is what makes it a next-word model. You also learn a quick test that catches a leaking mask.

[Read the lesson](https://aicourses.dev/research/transformers/causal-masking)

### The mask

The fix is to hide every later position before the softmax. Take the table of scores, one row per token asking and one column per token answering, and set every cell above the diagonal to minus infinity. Since $e^{-\infty} = 0$, those cells get a share of exactly 0, and softmax shares the full 100% among the tokens that are allowed.

In [ ]:
T = 4
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
print(mask.int())
scores = torch.zeros(T, T)          # pretend every score is 0
scores = scores.masked_fill(~mask, float("-inf"))
print(F.softmax(scores, dim=-1).round(decimals=2))

Here is the head from the last lesson with the mask added:

In [ ]:
def head(x, Wq, Wk, Wv, causal=True):
    q, k, v = x @ Wq, x @ Wk, x @ Wv
    scores = q @ k.T / k.shape[-1] ** 0.5
    if causal:                      # hide every later token
        T = x.shape[0]
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float("-inf"))
    return F.softmax(scores, dim=-1) @ v

### A test that catches a leak

A leaking mask raises no error. Training runs, the loss falls beautifully, and nothing looks wrong until the model writes. So test the property directly: changing a later token must not change any earlier output.

In [ ]:
torch.manual_seed(42)
x = torch.randn(4, 8)
Wq, Wk, Wv = [torch.randn(8, 4) / 8 ** 0.5 for _ in range(3)]
x2 = x.clone()
x2[3] = torch.randn(8)              # change only the last token
for causal in [True, False]:
    a = head(x, Wq, Wk, Wv, causal)
    b = head(x2, Wq, Wk, Wv, causal)
    print(causal, (a[:3] - b[:3]).abs().max().item())
# True 0.0
# False 0.520298421382904

## 10.4 Multi-head attention and projections

Several attention heads run side by side, and each looks for a different kind of link between words. Splitting the data into heads and joining it back is a reshape you will write many times, and a task that can come up in interviews.

[Read the lesson](https://aicourses.dev/research/transformers/multi-head-attention)

### Splitting into heads

Here it is with small numbers: a batch of 2 sequences, 5 tokens each, 16 channels and 4 heads, so a head size of 4. The code calls them `B`, `T`, `C` and `nh`, the usual names; this `C` is the number of channels, not lesson 9.3’s table of embeddings.

In [ ]:
torch.manual_seed(42)
B, T, C, nh = 2, 5, 16, 4        # batch, tokens, channels, heads
hs = C // nh                     # head size: 4
x = torch.randn(B, T, C)
qkv = nn.Linear(C, 3 * C)        # all three projections in one
q, k, v = qkv(x).split(C, dim=2)            # each (2, 5, 16)
q = q.view(B, T, nh, hs).transpose(1, 2)    # (2, 4, 5, 4)
k = k.view(B, T, nh, hs).transpose(1, 2)
v = v.view(B, T, nh, hs).transpose(1, 2)
att = q @ k.transpose(-2, -1) / hs ** 0.5   # (2, 4, 5, 5)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)
y = att @ v                                 # (2, 4, 5, 4)
y = y.transpose(1, 2).reshape(B, T, C)      # (2, 5, 16)
print(q.shape, att.shape, y.shape)

PyTorch has this computation built in, as `F.scaled_dot_product_attention`, with `is_causal=True` applying the mask, and on a GPU it can use much faster methods that you meet in Module 11. Check yours against it:

In [ ]:
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
ref = ref.transpose(1, 2).reshape(B, T, C)
print((y - ref).abs().max().item())   # 8.940696716308594e-08

A difference of about one ten-millionth is rounding. Now the classic bug: skip the transpose and reshape straight to the final shape.

In [ ]:
q2, k2, v2 = qkv(x).split(C, dim=2)
wrong = q2.reshape(B, nh, T, hs)            # skips the transpose
print((wrong - q).abs().max().item())       # 3.0071001052856445

### Projections in and out

There are two learned matrices around the heads. The first, here one `nn.Linear` from C to 3C, makes every head’s queries, keys and values in one go. The second, the **output projection**, takes the joined heads’ outputs and mixes them into one result per token, so what one head found can be combined with what another found. In the book club, it is the meeting where the notes are merged.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)   # mixes the heads

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        split = lambda t: t.view(B, T, self.n_head, -1).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            split(q), split(k), split(v), is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, C)  # join the heads
        return self.proj(y)

attn = CausalSelfAttention(768, 12)          # GPT-2 small's sizes
print(sum(p.numel() for p in attn.parameters()))   # 2362368

## 10.5 Positional information: from learned positions to RoPE

Attention on its own does not know word order, so each word must carry its position. RoPE does this by turning each word’s vector by an angle set by its position. It is how modern models handle long texts.

[Read the lesson](https://aicourses.dev/research/transformers/positions-and-rope)

### Attention is blind to order

Take three tokens, run them through one head without the mask, then run them again in reverse order and compare.

In [ ]:
torch.manual_seed(42)
x = torch.randn(3, 8)            # "dog", "bites", "man"
Wq, Wk, Wv = [torch.randn(8, 4) / 8 ** 0.5 for _ in range(3)]
order = [2, 1, 0]                # "man", "bites", "dog"
a = head(x, Wq, Wk, Wv, causal=False)
b = head(x[order], Wq, Wk, Wv, causal=False)
print((a[order] - b).abs().max().item())   # 2.384185791015625e-07

### Learned positions

GPT-2’s fix is simple. Next to the table of token embeddings, keep a second table with one learned vector per position: one for position 0, one for position 1, and so on up to 1,023. Add a token’s position vector to its token vector before the first layer. Now “dog” at position 0 and “dog” at position 2 start as different lists of numbers.

In [ ]:
vocab_size, n_embd, block_size = 65, 32, 64   # small example sizes
idx = torch.randint(vocab_size, (2, 8))         # 2 texts of 8 tokens
B, T = idx.shape
wte = nn.Embedding(vocab_size, n_embd)   # one row per token
wpe = nn.Embedding(block_size, n_embd)   # one row per position
x = wte(idx) + wpe(torch.arange(T))      # (B, T, n_embd)

### RoPE in code

In [ ]:
def rope(x, pos, base=10000.0):
    d = x.shape[-1]                       # must be even
    freqs = base ** (-torch.arange(0, d, 2) / d)  # one per pair
    ang = pos[:, None] * freqs[None, :]   # (T, d/2) angles
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]   # the two halves of pairs
    turned = torch.stack([x1 * cos - x2 * sin,
                          x1 * sin + x2 * cos], dim=-1)
    return turned.flatten(-2)

`x[..., 0::2]` takes the numbers in even places and `x[..., 1::2]` those in odd places, so each pair is one even and one odd number side by side. `pos[:, None]` and `freqs[None, :]` add a dimension of size 1, so broadcasting multiplies every position by every pair’s speed, and `flatten(-2)` merges the last two dimensions, putting each turned pair back side by side. Now test the promise: the score should depend only on the gap.

In [ ]:
torch.manual_seed(0)
q, k = torch.randn(1, 64), torch.randn(1, 64)
for m, n in [(5, 3), (10, 8), (1005, 1003), (5, 4)]:
    qm = rope(q, torch.tensor([float(m)]))
    kn = rope(k, torch.tensor([float(n)]))
    print(m, n, round((qm @ kn.T).item(), 4))
# 5 3 -10.1427
# 10 8 -10.1427
# 1005 1003 -10.1427
# 5 4 -11.4563

## 10.6 The transformer block: residuals, layer norm, MLP

A block is attention plus a small network, with a shortcut that adds the block’s input to its output. That shortcut, called the residual, lets very deep stacks train. In Module 15 it is how you look inside a model.

[Read the lesson](https://aicourses.dev/research/transformers/transformer-block)

### The shortcut: add, don’t replace

See it in a stack of 30 small layers, each a linear layer followed by tanh, with and without the shortcut:

In [ ]:
def grads(residual, depth=30):
    torch.manual_seed(42)
    layers = [nn.Linear(32, 32) for _ in range(depth)]
    x = torch.randn(64, 32)
    for layer in layers:
        out = torch.tanh(layer(x))
        x = x + out if residual else out   # the shortcut, or not
    x.pow(2).mean().backward()
    first = layers[0].weight.grad.norm().item()
    last = layers[-1].weight.grad.norm().item()
    return f"first {first:.1e}  last {last:.1e}"

print("plain:   ", grads(residual=False))
print("residual:", grads(residual=True))
# plain:    first 7.0e-10  last 3.0e-02
# residual: first 6.3e+00  last 4.5e+00

### Layer norm, before each branch

Work it through on one token with the four numbers 1, 2, 3 and 4. The mean is 2.5. The variance is the average squared distance from the mean, (2.25 + 0.25 + 0.25 + 2.25) / 4 = 1.25, so the standard deviation is about 1.118. Each number becomes its distance from the mean divided by 1.118: −1.342, −0.447, 0.447 and 1.342. PyTorch agrees:

In [ ]:
ln = nn.LayerNorm(4)
print(ln(torch.tensor([1.0, 2.0, 3.0, 4.0])).detach())
# tensor([-1.3416, -0.4472,  0.4472,  1.3416])

### The small network in every block

In GPT-2 it widens each token’s 768 numbers to 3,072, four times as many, applies an activation, then narrows back to 768. The activation is **GELU**, a smooth cousin of ReLU that curves gently near zero instead of bending sharply.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.fc = nn.Linear(n_embd, 4 * n_embd)     # widen
        self.proj = nn.Linear(4 * n_embd, n_embd)   # narrow back

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

### The block

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # read, then add back
        x = x + self.mlp(self.ln2(x))
        return x

block = Block(768, 12)                   # GPT-2 small's sizes
print(sum(p.numel() for p in block.parameters()))   # 7087872

## 10.7 Training GPT on Tiny Shakespeare

Assembles the block and trains it with Module 8’s loop and checks, including a loss at initialization close to ln(vocabulary size).

[Read the lesson](https://aicourses.dev/research/transformers/training-gpt)

### The data

The model reads characters, so its vocabulary is every distinct character in the file.

In [ ]:
import math
import urllib.request

url = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
       "master/data/tinyshakespeare/input.txt")
urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt").read()
chars = sorted(set(text))                 # every symbol used
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[ch] for ch in text])
n = int(0.9 * len(data))
train, val = data[:n], data[n:]           # first 90% to train
print(len(text), len(chars))              # 1115394 65

A training example is a window of 64 characters, and its answer is the same window shifted along by one. From lesson 10.3, one window teaches 64 next-character guesses at once.

In [ ]:
block_size, batch_size = 64, 32
gen = torch.Generator().manual_seed(42)   # for training batches

def get_batch(split, g=gen):
    d = train if split == "train" else val
    ix = torch.randint(len(d) - block_size, (batch_size,),
                       generator=g)
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x, y                           # y is x shifted by one

### The model

A GPT is a token table and a position table, a stack of blocks from lesson 10.6, a last layer norm, and a linear layer, called the head here (nothing to do with attention heads), that turns each token’s numbers into 65 scores. The weights start small and random, with a standard deviation of 0.02, as in GPT-2.

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab, block_size, n_layer, n_head, n_embd):
        super().__init__()
        self.wte = nn.Embedding(vocab, n_embd)       # tokens
        self.wpe = nn.Embedding(block_size, n_embd)  # positions
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab, bias=False)
        self.apply(self._init)

    def _init(self, m):                  # GPT-2's starting weights
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.wte(idx) + self.wpe(torch.arange(T))  # (B, T, C)
        return self.head(self.ln_f(self.blocks(x)))    # (B, T, V)

### Three checks before you train

In [ ]:
def loss_on(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, logits.size(-1)),
                           y.view(-1))

torch.manual_seed(42)
model = GPT(len(chars), block_size, n_layer=4, n_head=4, n_embd=128)
print(sum(p.numel() for p in model.parameters()))   # 818176
x, y = get_batch("train")
print(round(loss_on(model, x, y).item(), 3),
      round(math.log(len(chars)), 3))               # 4.138 4.174

**Check 2: no peeking.** Change the last character of every window; no earlier position’s scores may move.

In [ ]:
x2 = x.clone()
x2[:, -1] = (x2[:, -1] + 1) % len(chars)  # change the last token
with torch.no_grad():
    diff = (model(x)[:, :-1] - model(x2)[:, :-1]).abs().max()
print(diff.item())                        # 0.0

**Check 3: it can learn at all.** Train a fresh model on this one batch, again and again. With 818,176 parameters it should memorise 32 windows easily. If the loss will not fall, something is broken.

In [ ]:
torch.manual_seed(0)
tiny = GPT(len(chars), block_size, n_layer=4, n_head=4, n_embd=128)
opt = torch.optim.AdamW(tiny.parameters(), lr=1e-3)
for step in range(200):                   # one batch, over and over
    loss = loss_on(tiny, x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(round(loss.item(), 3))              # 0.019

### The run

Now Module 8’s loop: AdamW, a little weight decay, gradients clipped at 1.0, and an estimate of the training and validation loss every 500 steps, averaged over 50 batches.

In [ ]:
egen = torch.Generator().manual_seed(1)   # for evaluation batches

@torch.no_grad()
def evaluate(split, batches=50):
    model.eval()
    total = sum(loss_on(model, *get_batch(split, egen)).item()
                for _ in range(batches))
    model.train()
    return total / batches

opt = torch.optim.AdamW(model.parameters(), lr=1e-3,
                        weight_decay=0.1)
for step in range(3001):
    if step % 500 == 0:
        print(step, round(evaluate("train"), 3),
              round(evaluate("val"), 3))
    if step == 3000:
        break
    x, y = get_batch("train")
    loss = loss_on(model, x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

### What it writes

Sampling works as in Module 9, except that the model now scores the whole window and you use the scores at the last position:

In [ ]:
@torch.no_grad()
def generate(idx, n, temperature=1.0, generator=None):
    for _ in range(n):
        logits = model(idx[:, -block_size:])[:, -1] / temperature
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1, generator=generator)
        idx = torch.cat([idx, nxt], dim=1)
    return idx

g = torch.Generator().manual_seed(42)
start = torch.zeros((1, 1), dtype=torch.long)   # a newline
out = generate(start, 300, generator=g)
print("".join(chars[i] for i in out[0].tolist()))

*The lesson shows:*

```text
Mistrous shadows, and not the nost to tnough countery
Bear heed lamp on, the crave to our pland hell
unto this u
ceavier famen, for the noble leadiness?

Lady Must Derby:
Plead here, though he hast? dost he should should
```

## 10.8 Tokenization: how text is cut into tokens, with byte-pair encoding

Real models read neither letters nor whole words but pieces of words called tokens. Byte-pair encoding (BPE) learns those pieces by merging the most common pairs. It explains odd spelling, odd arithmetic and why some languages cost more.

[Read the lesson](https://aicourses.dev/research/transformers/tokenization-and-bpe)

### Letters, words or something in between

Start from bytes. A computer stores text as UTF-8, where each character becomes one to four **bytes**, numbers from 0 to 255:

In [ ]:
for s in ["hi", "é", "न", "🙂"]:
    print(repr(s), list(s.encode("utf-8")))
# 'hi' [104, 105]
# 'é' [195, 169]
# 'न' [224, 164, 168]
# '🙂' [240, 159, 153, 130]

### BPE in code

Two helpers: count every neighbouring pair, and replace one pair with its new token.

In [ ]:
def pair_counts(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):          # neighbours
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)              # the pair becomes one
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

Now train 200 merges on Tiny Shakespeare, the `input.txt` that lesson 10.7 downloaded; in a new Colab runtime, run its download lines again first, because a new runtime starts without your files. New tokens get the numbers 256, 257 and so on, and `vocab` remembers which bytes each one stands for.

In [ ]:
text = open("input.txt").read()
ids = list(text.encode("utf-8"))
vocab = {i: bytes([i]) for i in range(256)}   # id -> its bytes
merges = {}                                   # pair -> new id
for new_id in range(256, 256 + 200):
    counts = pair_counts(ids)
    pair = max(counts, key=counts.get)        # most common pair
    ids = merge(ids, pair, new_id)
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
print(len(text.encode("utf-8")), len(ids))    # 1115394 598357

To encode new text, apply the learned merges in the order they were learned; to decode, glue the bytes back together:

In [ ]:
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        counts = pair_counts(ids)
        # the pair that was merged earliest in training goes first
        pair = min(counts, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                             # nothing left to merge
        ids = merge(ids, pair, merges[pair])
    return ids

def decode(ids):
    raw = b"".join(vocab[i] for i in ids)
    return raw.decode("utf-8", errors="replace")

A tokenizer has one promise to keep: decoding what you encoded gives back exactly the original text. Test it on the awkward cases, not only on Shakespeare:

In [ ]:
tests = ["", "To be, or not to be", "naïve café", "नमस्ते", "🙂🙂"]
for s in tests:
    assert decode(encode(s)) == s, s
print("all round trips pass")

### The real thing: GPT-2’s tokenizer

You can load the real tokenizer with Hugging Face Transformers. It downloads a few small files the first time:

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
print(len(tok))                          # 50257
for s in ["strawberry", " strawberry", "12345 + 67890"]:
    ids = tok.encode(s)
    print(repr(s), len(ids), tok.convert_ids_to_tokens(ids))
# 'strawberry' 3 ['st', 'raw', 'berry']
# ' strawberry' 1 ['Ġstrawberry']
# '12345 + 67890' 6 ['123', '45', 'Ġ+', 'Ġ6', '78', '90']

## 10.9 Modern upgrades: RMSNorm, SwiGLU and a map of mixture of experts

Today’s open models, such as Meta’s Llama, change your block in small ways: a simpler normalization (RMSNorm) and a better small network (SwiGLU). You add both. Then you see, as a map, how mixture of experts runs only part of the model for each token.

[Read the lesson](https://aicourses.dev/research/transformers/modern-upgrades)

### RMSNorm: normalize without the mean

Work it on 1, 2, 3 and 4. The squares are 1, 4, 9 and 16, their mean is 7.5, and the RMS, its square root, is about 2.739. Divide each number by it: 0.365, 0.730, 1.095 and 1.461. Unlike layer norm, the mean is not moved to zero; only the size is fixed. In the authors’ experiments, it matched layer norm’s quality while cutting running time by 7% to 64%, depending on the model.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # learned scale

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / rms

x = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(RMSNorm(4)(x).detach())
print(nn.RMSNorm(4, eps=1e-6)(x).detach())
# tensor([0.3651, 0.7303, 1.0954, 1.4606])
# tensor([0.3651, 0.7303, 1.0954, 1.4606])

### SwiGLU: a network with a dimmer switch

Three matrices instead of two would add parameters, so the hidden size shrinks to two thirds of 4 × C, the choice described in the first LLaMA paper. With C = 768, the hidden size becomes 2,048:

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        hidden = int(2 / 3 * 4 * n_embd)       # keeps the size equal
        self.gate = nn.Linear(n_embd, hidden, bias=False)
        self.up = nn.Linear(n_embd, hidden, bias=False)
        self.down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

mlp = nn.Sequential(nn.Linear(768, 3072, bias=False), nn.GELU(),
                    nn.Linear(3072, 768, bias=False))
print(sum(p.numel() for p in mlp.parameters()),
      sum(p.numel() for p in SwiGLU(768).parameters()))
# 4718592 4718592

### A Llama-style block

Swap both parts into lesson 10.6’s block, and move RoPE from lesson 10.5 into attention, turning the queries and keys just before the dot product. Following Llama’s published code, the linear layers drop their biases.

In [ ]:
class RopeAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        split = lambda t: t.view(B, T, self.n_head, -1).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        pos = torch.arange(T, dtype=torch.float32)
        q, k = rope(q, pos), rope(k, pos)       # turn by position
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

The block puts the pieces together, each with its own RMSNorm in front:

In [ ]:
class LlamaBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = RopeAttention(n_embd, n_head)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

## 10.10 How to read a paper, starting with the one you just built

A research paper can look impossible at first. You learn to read one in passes, on “Attention Is All You Need”, the paper behind the model you just built, so every equation in it is one you have already coded.

[Read the lesson](https://aicourses.dev/research/transformers/reading-papers)

### Pass 2: every equation, against your code

Concat means joining the heads side by side, Wᴼ is the output projection, and Sublayer stands for attention or the MLP. Two of these are worth running, because running code is a check on your reading of the paper. The sine-wave positions: even places get $\sin(pos / 10000^{2i/d})$ and odd places $\cos$ of the same angle, where $pos$ is the position, $i$ the pair number and $d$ the numbers per token.

In [ ]:
def sinusoidal(n_pos, d):
    pos = torch.arange(n_pos, dtype=torch.float32)[:, None]
    i = torch.arange(0, d, 2, dtype=torch.float32)       # 2i
    angle = pos / 10000 ** (i / d)
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2] = torch.sin(angle)      # even places: sine
    pe[:, 1::2] = torch.cos(angle)      # odd places: cosine
    return pe

print(sinusoidal(4, 6).round(decimals=3))

And equation 3, the learning rate, where $d_{\text{model}} = 512$ and warmup is 4,000 steps:

In [ ]:
def lrate(step, d_model=512, warmup=4000):
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

for step in [1, 1000, 4000, 16000, 100000]:
    print(step, f"{lrate(step):.2e}")
# 1 1.75e-07
# 1000 1.75e-04
# 4000 6.99e-04
# 16000 3.49e-04
# 100000 1.40e-04

## 10.11 Beyond text: images as tokens

Pictures can be cut into small squares and treated like tokens. This optional lesson shows how models that read both images and text are put together, as many big open models now do.

[Read the lesson](https://aicourses.dev/research/transformers/beyond-text)

### Patches in code

`unfold` cuts a tensor into windows along one dimension. Applied to height and then width, it cuts the image into a 14 × 14 grid of 16 × 16 patches:

In [ ]:
torch.manual_seed(0)
image = torch.rand(3, 224, 224)     # colour, height, width
p = 16                              # patch size in pixels
grid = image.unfold(1, p, p).unfold(2, p, p)   # (3, 14, 14, 16, 16)
patches = grid.permute(1, 2, 0, 3, 4).reshape(-1, 3 * p * p)
print(grid.shape, patches.shape)
# torch.Size([3, 14, 14, 16, 16]) torch.Size([196, 768])
same = torch.equal(patches[0], image[:, :16, :16].reshape(-1))
print(same)                         # True

Now project each patch to a token. Most real code does it with a **convolution**, a layer that slides one small learned window across the image and multiplies every spot it lands on by the same weights. With the window and its step both equal to the patch size, that is the same thing:

In [ ]:
embed = nn.Linear(3 * p * p, 768)   # one learned projection
tokens = embed(patches)             # (196, 768): one token per patch
conv = nn.Conv2d(3, 768, kernel_size=p, stride=p)
with torch.no_grad():
    conv.weight.copy_(embed.weight.view(768, 3, p, p))
    conv.bias.copy_(embed.bias)
    via_conv = conv(image).flatten(1).T            # (196, 768)
print(tokens.shape, (tokens - via_conv).abs().max().item())
# torch.Size([196, 768]) 0.0